<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_29_Windows_Event_Log_Brute_Force_Login_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Experiment 1: Windows Event Log Brute-Force Login Detector
# ============================================

from datetime import datetime, timedelta

# Function to convert string to datetime
def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")


# Function to detect brute-force attacks
def detect_bruteforce(events, threshold=5, window_minutes=2):
    """
    Detect brute-force login attempts:
    - At least 'threshold' failed logins (Event ID 4625)
    - Within 'window_minutes'
    - Followed by a successful login (Event ID 4624)
    """

    # Sort events by timestamp
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))

    # Group events by account
    by_account = {}

    for event in events:
        by_account.setdefault(event["account"], []).append(event)

    results = {}

    # Check each account
    for account, account_events in by_account.items():

        failures = [e for e in account_events if e["event_id"] == 4625]
        successes = [e for e in account_events if e["event_id"] == 4624]

        flagged = False

        # Sliding window over failed logins
        for i in range(len(failures)):

            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)

            count = sum(
                1
                for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )

            if count >= threshold:
                flagged = True
                break

        if flagged:

            followed_by_success = (
                len(successes) > 0 and
                any(
                    parse_time(s["timestamp"]) >
                    parse_time(failures[-1]["timestamp"])
                    for s in successes
                )
            )

            results[account] = {
                "failed_attempts": len(failures),
                "followed_by_success": followed_by_success,
                "source_ips": sorted(
                    {f["source_ip"] for f in failures}
                )
            }

    return results


# ============================================
# Test Cases
# ============================================

def test_experiment1():

    events = []

    base = datetime(2026, 1, 15, 3, 40, 0)

    # Six failed Administrator logins
    for i in range(6):
        events.append({
            "event_id": 4625,
            "account": "Administrator",
            "timestamp": (
                base + timedelta(seconds=15 * i)
            ).strftime("%Y-%m-%d %H:%M:%S"),
            "source_ip": "203.0.113.7"
        })

    # Successful Administrator login
    events.append({
        "event_id": 4624,
        "account": "Administrator",
        "timestamp": (
            base + timedelta(seconds=100)
        ).strftime("%Y-%m-%d %H:%M:%S"),
        "source_ip": "203.0.113.7"
    })

    # Normal user login
    events.append({
        "event_id": 4624,
        "account": "jsmith",
        "timestamp": "2026-01-15 09:00:00",
        "source_ip": "10.0.0.5"
    })

    results = detect_bruteforce(events)

    # Display Results
    print("Detected Accounts:")
    print(results)
    print()

    # Assertions
    assert "Administrator" in results
    assert results["Administrator"]["failed_attempts"] == 6
    assert results["Administrator"]["followed_by_success"] == True
    assert results["Administrator"]["source_ips"] == ["203.0.113.7"]
    assert "jsmith" not in results

    print("All test cases passed.")


# Run Test
test_experiment1()

Detected Accounts:
{'Administrator': {'failed_attempts': 6, 'followed_by_success': True, 'source_ips': ['203.0.113.7']}}

All test cases passed.
